# Zeinab Dast Mozd (2323135)
# CF969-7-SU Big Data for Computational Finance
# Assignment 2 : Code

### First method of filling missing values
### A function that do preprocessing and preparing to machine learning models

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, Normalizer
from sklearn.compose import ColumnTransformer

def preprocess_train_data(train_data):
    # Create target variable
    train_data['y'] = np.where(train_data['loan_status'] == 'Charged Off', 1, 0)
    print("The shape of the train data is ", train_data.shape)
    
    # Separate features and target and dropping loan status column
    X = train_data.drop(['loan_status', 'y'], axis=1)
    y = train_data['y']
    
    # Check for NaN values in data
    nan_count = X.isnull().sum()
    print("Number of NaN values in X:", nan_count)
    
    # Fill numerical columns with median
    numerical_columns = ['dti', 'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq', 
                         'open_acc', 'pub_rec', 'revol_util', 'total_acc', 
                         'collections_12_mths_ex_med', 'acc_now_delinq', 
                         'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'annual_inc']

    imputer = SimpleImputer(strategy='median')
    X[numerical_columns] = imputer.fit_transform(X[numerical_columns])
    
    # Fill 'emp_length' with mode
    X['emp_length'] = X['emp_length'].fillna(X['emp_length'].mode()[0])
    
    # Drop columns 'id' and 'member_id'
    columns_to_drop = ['id', 'member_id']
    X = X.drop(columns=columns_to_drop)
    
    # Print the columns of X after dropping
    print("Columns of X after dropping 'id' and 'member_id':")
    print(X.columns)

    # Check for NaN values in data
    nan_count = X.isnull().sum()
    print("Number of NaN values in X:", nan_count)
    
    
       # Identify categorical and numerical features
    categorical_features = X.select_dtypes(include=['object']).columns
    numerical_features = X.select_dtypes(include=[np.number]).columns
    
    print(f"Categorical features count: {len(categorical_features)}")
    print(f"Numerical features count: {len(numerical_features)}")
    
    # One-hot encode categorical features
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_categorical = encoder.fit_transform(X[categorical_features])
    
    # Convert numerical features to numpy array
    X_numerical = X[numerical_features].to_numpy()
    
    # Combine numerical and categorical features
    X_combined = np.hstack((X_numerical, X_categorical))
    
    # Normalize all features
    scaler = StandardScaler()
    X_processed = scaler.fit_transform(X_combined)
    
    print(f"Shape of processed X after normalization: {X_processed.shape}")
    
    # Assuming y does not need any preprocessing, you can directly return X_processed and y
    return X_processed, y



### load the train data and call the previous function to prepare the data for modelling

In [2]:
# Load the train data
train_data = pd.read_csv('trainData.csv')

# Call the preprocess_train_data function
X_processed, y = preprocess_train_data(train_data)

# Print the processed features and target variable
print("Processed features (X):", X_processed)
print("Target variable (y):", y)
# Print the number of columns in the processed train data
print("Number of columns in processed train features:", X_processed.shape[1])

The shape of the train data is  (226067, 34)
Number of NaN values in X: id                            226067
member_id                     226067
loan_amnt                          0
int_rate                           0
installment                        0
grade                              0
emp_length                     14611
home_ownership                     0
annual_inc                         0
dti                              180
delinq_2yrs                        5
inq_last_6mths                     5
mths_since_last_delinq        115840
open_acc                           5
pub_rec                            5
revol_bal                          0
revol_util                       190
total_acc                          5
total_pymnt                        0
total_pymnt_inv                    0
total_rec_prncp                    0
total_rec_int                      0
total_rec_late_fee                 0
recoveries                         0
collection_recovery_fee            0
las

### load the test data and call the previous function to prepare the data for modelling

In [3]:
# Load the test data
test_data = pd.read_csv('testdata.csv')

# Call the preprocess_data function for test data
X_test_processed, y_test = preprocess_train_data(test_data)

# Print the processed features and target variable for test data
print("Processed test features (X_test):", X_test_processed)
print("Test target variable (y_test):", y_test)

The shape of the train data is  (226067, 34)
Number of NaN values in X: id                            226067
member_id                     226067
loan_amnt                          0
int_rate                           0
installment                        0
grade                              0
emp_length                     14799
home_ownership                     0
annual_inc                         1
dti                              174
delinq_2yrs                        5
inq_last_6mths                     6
mths_since_last_delinq        115999
open_acc                           5
pub_rec                            5
revol_bal                          0
revol_util                       159
total_acc                          5
total_pymnt                        0
total_pymnt_inv                    0
total_rec_prncp                    0
total_rec_int                      0
total_rec_late_fee                 0
recoveries                         0
collection_recovery_fee            0
las

### LinearRegression model

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Assuming you have X_train, X_test, y_train, and y_test prepared
# X_train and X_test should contain all predictor variables

# Step 1: Fit the linear regression model
lr_model = LinearRegression()
lr_model.fit(X_processed, y)

# Step 2: Predict on training data
y_pred_train = lr_model.predict(X_processed)

# Step 3: Calculate MSE for training data
mse_train = mean_squared_error(y, y_pred_train)
print(f"Mean Squared Error (Training data): {mse_train}")

# Step 4: Predict on testing data
y_pred_test = lr_model.predict(X_test_processed)

# Step 5: Calculate MSE for testing data
mse_test = mean_squared_error(y_test, y_pred_test)
print(f"Mean Squared Error (Testing data): {mse_test}")


Mean Squared Error (Training data): 0.06780941597891035
Mean Squared Error (Testing data): 1.693199897110197e+18


### Second method of filling missing values
### A function that do preprocessing and preparing to machine learning models

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, Normalizer

def preprocess_data(train_data):
    # Create target variable
    train_data['y'] = np.where(train_data['loan_status'] == 'Charged Off', 1, 0)
    print("The shape of the data is ", train_data.shape)
    print(train_data.head())
    
    # Check for NaN values in dataset
    print("Number of NaN values in X:", train_data.isnull().sum())
    
    # Drop columns 'id', 'member_id', and 'mths_since_last_delinq'
    columns_to_drop = ['id', 'member_id', 'mths_since_last_delinq']
    train_data = train_data.drop(columns=columns_to_drop)
    
    # Print the columns of X after dropping
    print("Columns of X after dropping 'id' and 'member_id':")
    print(train_data.columns)
    
    # Remove rows with NaN values
    train_data = train_data.dropna()
    
    # Print the cleaned DataFrame shape
    print("DataFrame after removing rows with NaN values:")
    print(train_data.shape)
    
    # Separate features and target, and drop 'loan_status' column
    y = train_data['y']
    X = train_data.drop(['loan_status', 'y'], axis=1)
    
    # Identify categorical and numerical features
    categorical_features = X.select_dtypes(include=['object']).columns
    numerical_features = X.select_dtypes(include=[np.number]).columns
    
    print(f"Categorical features count: {len(categorical_features)}")
    print(f"Numerical features count: {len(numerical_features)}")
    
    # One-hot encode categorical features
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_categorical = encoder.fit_transform(X[categorical_features])
    
    # Convert numerical features to numpy array
    X_numerical = X[numerical_features].to_numpy()
    
    # Combine numerical and categorical features
    X_combined = np.hstack((X_numerical, X_categorical))
    
    scaler = StandardScaler()
    X_processed = scaler.fit_transform(X_combined)
    
    print(f"Shape of processed X after normalization: {X_processed.shape}")
    
    # Return processed features and target variable
    return X_processed, y




### load the train data and call the previous function to prepare the data for modelling

In [6]:
# Load the train data
train_data = pd.read_csv('trainData.csv')

# Call the preprocess_train_data function
X_processed, y = preprocess_data(train_data)

# Print the processed features and target variable
#print("Processed features (X):", X_processed)
#print("Target variable (y):", y)
# Print the number of columns in the processed train data
#print("Number of columns in processed train features:", X_processed.shape[1])

The shape of the data is  (226067, 34)
   id  member_id  loan_amnt  int_rate  installment grade emp_length  \
0 NaN        NaN      18600     10.99       608.86     B    6 years   
1 NaN        NaN       2000     17.97        72.28     D    4 years   
2 NaN        NaN      12000     12.29       400.24     C  10+ years   
3 NaN        NaN      16000     19.42       589.90     D    7 years   
4 NaN        NaN      22525     16.02       548.01     C  10+ years   

  home_ownership  annual_inc loan_status  ...  recoveries  \
0           RENT     80000.0  Fully Paid  ...         0.0   
1       MORTGAGE     55400.0     Current  ...         0.0   
2            OWN     60000.0  Fully Paid  ...         0.0   
3           RENT     64000.0     Current  ...         0.0   
4       MORTGAGE     94080.0  Fully Paid  ...         0.0   

   collection_recovery_fee  last_pymnt_amnt  collections_12_mths_ex_med  \
0                      0.0         15705.09                         0.0   
1                

### load the test data and call the previous function to prepare the data for modelling

In [7]:
# Load the test data
test_data = pd.read_csv('testdata.csv')

# Call the preprocess_data function for test data
X_test_processed, y_test = preprocess_data(test_data)

# Print the processed features and target variable for test data
print("Processed test features (X_test):", X_test_processed)
print("Test target variable (y_test):", y_test)

The shape of the data is  (226067, 34)
   id  member_id  loan_amnt  int_rate  installment grade emp_length  \
0 NaN        NaN       8000      7.07       247.28     A  10+ years   
1 NaN        NaN      20000      7.21       619.47     A    3 years   
2 NaN        NaN      20000     12.74       452.41     C    5 years   
3 NaN        NaN      20000      8.81       634.23     A  10+ years   
4 NaN        NaN      20000     15.31       479.06     C  10+ years   

  home_ownership  annual_inc loan_status  ...  recoveries  \
0       MORTGAGE     78000.0  Fully Paid  ...         0.0   
1           RENT     78000.0     Current  ...         0.0   
2       MORTGAGE     97000.0     Current  ...         0.0   
3       MORTGAGE    115000.0  Fully Paid  ...         0.0   
4            OWN     75000.0  Fully Paid  ...         0.0   

   collection_recovery_fee  last_pymnt_amnt  collections_12_mths_ex_med  \
0                      0.0          6844.48                         0.0   
1                

### LinearRegression model

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Assuming you have X_train, X_test, y_train, and y_test prepared
# X_train and X_test should contain all predictor variables

# Step 1: Fit the linear regression model
lr_model = LinearRegression()
lr_model.fit(X_processed, y)

# Step 2: Predict on training data
y_pred_train = lr_model.predict(X_processed)

# Step 3: Calculate MSE for training data
mse_train = mean_squared_error(y, y_pred_train)
print(f"Mean Squared Error (Training data): {mse_train}")

# Step 4: Predict on testing data
y_pred_test = lr_model.predict(X_test_processed)

# Step 5: Calculate MSE for testing data
mse_test = mean_squared_error(y_test, y_pred_test)
print(f"Mean Squared Error (Testing data): {mse_test}")


Mean Squared Error (Training data): 0.0654716056696797
Mean Squared Error (Testing data): 6.215507141425878e+16


### Ridge model

In [9]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Step 1: Initialize variables to track the best model and MSE
best_alpha = None
best_mse_train = float('inf')
best_mse_test = float('inf')

# Step 2: Loop over alpha values from 0.01 to 3.0 with an increment of 0.01
for alpha in np.arange(0.01, 3.01, 0.01):
    # Step 3: Fit the ridge regression model
    ridge_model = Ridge(alpha=alpha)
    ridge_model.fit(X_processed, y)

    # Step 4: Predict on training data
    y_pred_train = ridge_model.predict(X_processed)

    # Step 5: Calculate MSE for training data
    mse_train = mean_squared_error(y, y_pred_train)

    # Step 6: Predict on testing data
    y_pred_test = ridge_model.predict(X_test_processed)

    # Step 7: Calculate MSE for testing data
    mse_test = mean_squared_error(y_test, y_pred_test)

    # Step 8: Update the best model based on training MSE
    if mse_train < best_mse_train:
        best_mse_train = mse_train
        best_mse_test = mse_test
        best_alpha = alpha

# Step 9: Output the results
print(f"Best Alpha: {best_alpha}")
print(f"Mean Squared Error (Training data): {best_mse_train}")
print(f"Mean Squared Error (Testing data): {best_mse_test}")


Best Alpha: 0.01
Mean Squared Error (Training data): 0.06547170890739468
Mean Squared Error (Testing data): 0.06620607027686866


### Lasso model

In [10]:
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error

# Step 1: Initialize variables to track the best model and MSE
best_alpha = None
best_mse_train = float('inf')
best_mse_test = float('inf')

# Step 2: Loop over alpha values from 0.01 to 3.0 with an increment of 0.01
for alpha in np.arange(0.01, 3.01, 0.01):
    # Step 3: Fit the Lasso regression model
    lasso_model = Lasso(alpha=alpha)
    lasso_model.fit(X_processed, y)

    # Step 4: Predict on training data
    y_pred_train = lasso_model.predict(X_processed)

    # Step 5: Calculate MSE for training data
    mse_train = mean_squared_error(y, y_pred_train)

    # Step 6: Predict on testing data
    y_pred_test = lasso_model.predict(X_test_processed)

    # Step 7: Calculate MSE for testing data
    mse_test = mean_squared_error(y_test, y_pred_test)

    # Step 8: Update the best model based on training MSE
    if mse_train < best_mse_train:
        best_mse_train = mse_train
        best_mse_test = mse_test
        best_alpha = alpha

# Step 9: Output the results
print(f"Best Alpha: {best_alpha}")
print(f"Mean Squared Error (Training data): {best_mse_train}")
print(f"Mean Squared Error (Testing data): {best_mse_test}")


Best Alpha: 0.01
Mean Squared Error (Training data): 0.06762740839251773
Mean Squared Error (Testing data): 0.06840822181278264


### RandomForestRegressor model

In [11]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Step 1: Initialize variables to track the best model and MSE
best_mse_train = float('inf')
best_mse_test = float('inf')
best_n_estimators = None

# Step 2: Loop over a range of n_estimators to find the best model
for n_estimators in [10, 50, 100, 200]:  # Example values; you can adjust as needed
    # Step 3: Fit the Random Forest model
    rf_model = RandomForestRegressor(n_estimators=n_estimators, random_state=42)
    rf_model.fit(X_processed, y)

    # Step 4: Predict on training data
    y_pred_train = rf_model.predict(X_processed)

    # Step 5: Calculate MSE for training data
    mse_train = mean_squared_error(y, y_pred_train)

    # Step 6: Predict on testing data
    y_pred_test = rf_model.predict(X_test_processed)

    # Step 7: Calculate MSE for testing data
    mse_test = mean_squared_error(y_test, y_pred_test)

    # Step 8: Update the best model based on training MSE
    if mse_train < best_mse_train:
        best_mse_train = mse_train
        best_mse_test = mse_test
        best_n_estimators = n_estimators

# Step 9: Output the results
print(f"Best n_estimators: {best_n_estimators}")
print(f"Mean Squared Error (Training data): {best_mse_train}")
print(f"Mean Squared Error (Testing data): {best_mse_test}")

# Step 10: Variable importance
importances = rf_model.feature_importances_
feature_importance = dict(zip(range(X_processed.shape[1]), importances))

# Step 11: Sort and print variable importance
sorted_importance = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)
print("Feature Importance:")
for idx, importance in sorted_importance:
    print(f"Feature {idx}: {importance}")

Best n_estimators: 200
Mean Squared Error (Training data): 0.002713125795087582
Mean Squared Error (Testing data): 0.8867148883690429
Feature Importance:
Feature 17: 0.6527160290160995
Feature 1: 0.057489254630644505
Feature 14: 0.03532665838562353
Feature 2: 0.03190022550233457
Feature 19: 0.023833530757153398
Feature 12: 0.01746173039789053
Feature 13: 0.0174168885277601
Feature 15: 0.01560519726376806
Feature 23: 0.013627219698667052
Feature 4: 0.013491222588946214
Feature 0: 0.012801756119255422
Feature 10: 0.012739814697633857
Feature 9: 0.011350813250166332
Feature 3: 0.011119244711279813
Feature 24: 0.0110898466798271
Feature 11: 0.00980870077769843
Feature 7: 0.007935557273875987
Feature 16: 0.005628835360120077
Feature 6: 0.004465654068396177
Feature 22: 0.004378429661699374
Feature 8: 0.002968016437243349
Feature 28: 0.002726858397587553
Feature 5: 0.0026802016506047213
Feature 29: 0.001589979814849918
Feature 32: 0.0012448389978496192
Feature 27: 0.0012031705086433008
Featur

### MLPRegressor model

In [14]:
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, accuracy_score

# Step 2: Define the Neural Network model
nn_model = MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu', solver='adam', max_iter=200)

# Step 3: Train the model
nn_model.fit(X_processed, y)

# Step 4: Predict on training data
y_pred_train_nn = nn_model.predict(X_processed)

# Step 5: Calculate MSE for training data
mse_train_nn = mean_squared_error(y, y_pred_train_nn)
print(f"Mean Squared Error (Training data) for NN: {mse_train_nn}")

# Step 6: Predict on testing data
y_pred_test_nn = nn_model.predict(X_test_processed)

# Step 7: Calculate MSE for testing data
mse_test_nn = mean_squared_error(y_test, y_pred_test_nn)
print(f"Mean Squared Error (Testing data) for NN: {mse_test_nn}")
# Convert predictions to binary (e.g., using a threshold of 0.5)
threshold = 0.5  # Example threshold
y_pred_train_nn_binary = (y_pred_train_nn > threshold).astype(int)
y_pred_test_nn_binary = (y_pred_test_nn > threshold).astype(int)

# Assuming y11 and y22 are binary labels (0 or 1)
accuracy_train_nn = accuracy_score(y, y_pred_train_nn_binary)
accuracy_test_nn = accuracy_score(y_test, y_pred_test_nn_binary)

print(f"Accuracy (Training data) for NN: {accuracy_train_nn}")
print(f"Accuracy (Testing data) for NN: {accuracy_test_nn}")


Mean Squared Error (Training data) for NN: 0.025496184364339976
Mean Squared Error (Testing data) for NN: 0.031145905314154298
Accuracy (Training data) for NN: 0.9701976710049907
Accuracy (Testing data) for NN: 0.9642888603952352


### Third method of filling missing values
### A function that do preprocessing and preparing to machine learning models

In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, Normalizer

def preprocess_train_data2(train_data):
    # Specify columns to fill with zero
    columns_to_fill_with_zero = ['mths_since_last_delinq']  # Adjust as per your actual column name
    
    # Fill the specified columns with zero
    train_data[columns_to_fill_with_zero] = train_data[columns_to_fill_with_zero].fillna(0)
    
    # Drop columns 'id' and 'member_id'
    columns_to_drop = ['id', 'member_id']
    train_data = train_data.drop(columns=columns_to_drop)
    
    # Print the columns of X after dropping
    print("Columns of X after dropping 'id' and 'member_id':")
    print(train_data.columns)
    
    # Remove rows with NaN values
    train_data = train_data.dropna()
    
    # Print the cleaned DataFrame shape
    print("DataFrame after removing rows with NaN values:")
    print(train_data.shape)
    
    # Create target variable
    train_data['y'] = np.where(train_data['loan_status'] == 'Charged Off', 1, 0)
    
    # Separate features and target, and drop 'loan_status' column
    y = train_data['y']
    X = train_data.drop(['loan_status', 'y'], axis=1)
    
    # Identify categorical and numerical features
    categorical_features = X.select_dtypes(include=['object']).columns
    numerical_features = X.select_dtypes(include=[np.number]).columns
    
    print(f"Categorical features count: {len(categorical_features)}")
    print(f"Numerical features count: {len(numerical_features)}")
    
    # One-hot encode categorical features
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_categorical = encoder.fit_transform(X[categorical_features])
    
    # Convert numerical features to numpy array
    X_numerical = X[numerical_features].to_numpy()
    
    # Combine numerical and categorical features
    X_combined = np.hstack((X_numerical, X_categorical))
    
    # Normalize all features
    scaler = StandardScaler()
    X_processed = scaler.fit_transform(X_combined)
    
    print(f"Shape of processed X after normalization: {X_processed.shape}")
    
    # Return processed features and target variable
    return X_processed, y



### load the train data and call the previous function to prepare the data for modelling

In [16]:
# Example usage:
# Load the train data
train_data = pd.read_csv('traindata.csv')

# Call the preprocess_train_data function
X_processed, y = preprocess_train_data2(train_data)

# Print the processed features and target variable
print("Processed features (X):", X_processed)
print("Target variable (y):", y)


Columns of X after dropping 'id' and 'member_id':
Index(['loan_amnt', 'int_rate', 'installment', 'grade', 'emp_length',
       'home_ownership', 'annual_inc', 'loan_status', 'dti', 'delinq_2yrs',
       'inq_last_6mths', 'mths_since_last_delinq', 'open_acc', 'pub_rec',
       'revol_bal', 'revol_util', 'total_acc', 'total_pymnt',
       'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
       'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
       'last_pymnt_amnt', 'collections_12_mths_ex_med', 'application_type',
       'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim'],
      dtype='object')
DataFrame after removing rows with NaN values:
(204380, 31)
Categorical features count: 4
Numerical features count: 26
Shape of processed X after normalization: (204380, 52)
Processed features (X): [[ 0.34940743 -0.43691348  0.57341978 ...  1.22990361  0.23073606
  -0.23073606]
 [-1.44869255  1.00268915 -1.42413766 ... -0.81307185 -4.33395626
   4.33395626]
 

### load the test data and call the previous function to prepare the data for modelling

In [20]:
# Load the test data
test_data = pd.read_csv('testdata.csv')

# Call the preprocess_data function for test data
X_test_processed, y_test = preprocess_train_data2(test_data)

#print the processed features and target variable for test data
print("Processed test features (X_test):", X_test_processed)
print("Test target variable (y_test):", y_test)

Columns of X after dropping 'id' and 'member_id':
Index(['loan_amnt', 'int_rate', 'installment', 'grade', 'emp_length',
       'home_ownership', 'annual_inc', 'loan_status', 'dti', 'delinq_2yrs',
       'inq_last_6mths', 'mths_since_last_delinq', 'open_acc', 'pub_rec',
       'revol_bal', 'revol_util', 'total_acc', 'total_pymnt',
       'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
       'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
       'last_pymnt_amnt', 'collections_12_mths_ex_med', 'application_type',
       'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim'],
      dtype='object')
DataFrame after removing rows with NaN values:
(204334, 31)
Categorical features count: 4
Numerical features count: 26
Shape of processed X after normalization: (204334, 52)
Processed test features (X_test): [[-0.79817012 -1.24231193 -0.77231989 ... -0.81207828  0.23235882
  -0.23235882]
 [ 0.50128573 -1.21347635  0.61459438 ...  1.23140838  0.23235882
  -0.2

### LinearRegression model

In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Assuming you have X_train, X_test, y_train, and y_test prepared
# X_train and X_test should contain all predictor variables

# Step 1: Fit the linear regression model
lr_model = LinearRegression()
lr_model.fit(X_processed, y)

# Step 2: Predict on training data
y_pred_train = lr_model.predict(X_processed)

# Step 3: Calculate MSE for training data
mse_train = mean_squared_error(y, y_pred_train)
print(f"Mean Squared Error (Training data): {mse_train}")

# Step 4: Predict on testing data
y_pred_test = lr_model.predict(X_test_processed)

# Step 5: Calculate MSE for testing data
mse_test = mean_squared_error(y_test, y_pred_test)
print(f"Mean Squared Error (Testing data): {mse_test}")

Mean Squared Error (Training data): 0.06546840324699656
Mean Squared Error (Testing data): 2.061948554476924e+18


### Ridge model

In [22]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Step 1: Initialize variables to track the best model and MSE
best_alpha = None
best_mse_train = float('inf')
best_mse_test = float('inf')

# Step 2: Loop over alpha values from 0.01 to 3.0 with an increment of 0.01
for alpha in np.arange(0.01, 3.01, 0.01):
    # Step 3: Fit the ridge regression model
    ridge_model = Ridge(alpha=alpha)
    ridge_model.fit(X_processed, y)

    # Step 4: Predict on training data
    y_pred_train = ridge_model.predict(X_processed)

    # Step 5: Calculate MSE for training data
    mse_train = mean_squared_error(y, y_pred_train)

    # Step 6: Predict on testing data
    y_pred_test = ridge_model.predict(X_test_processed)

    # Step 7: Calculate MSE for testing data
    mse_test = mean_squared_error(y_test, y_pred_test)

    # Step 8: Update the best model based on training MSE
    if mse_train < best_mse_train:
        best_mse_train = mse_train
        best_mse_test = mse_test
        best_alpha = alpha

# Step 9: Output the results
print(f"Best Alpha: {best_alpha}")
print(f"Mean Squared Error (Training data): {best_mse_train}")
print(f"Mean Squared Error (Testing data): {best_mse_test}")


Best Alpha: 0.01
Mean Squared Error (Training data): 0.06546797170470291
Mean Squared Error (Testing data): 0.06620194985396584


### Lasso model

In [23]:
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error

# Step 1: Initialize variables to track the best model and MSE
best_alpha = None
best_mse_train = float('inf')
best_mse_test = float('inf')

# Step 2: Loop over alpha values from 0.01 to 3.0 with an increment of 0.01
for alpha in np.arange(0.01, 3.01, 0.01):
    # Step 3: Fit the Lasso regression model
    lasso_model = Lasso(alpha=alpha)
    lasso_model.fit(X_processed, y)

    # Step 4: Predict on training data
    y_pred_train = lasso_model.predict(X_processed)

    # Step 5: Calculate MSE for training data
    mse_train = mean_squared_error(y, y_pred_train)

    # Step 6: Predict on testing data
    y_pred_test = lasso_model.predict(X_test_processed)

    # Step 7: Calculate MSE for testing data
    mse_test = mean_squared_error(y_test, y_pred_test)

    # Step 8: Update the best model based on training MSE
    if mse_train < best_mse_train:
        best_mse_train = mse_train
        best_mse_test = mse_test
        best_alpha = alpha

# Step 9: Output the results
print(f"Best Alpha: {best_alpha}")
print(f"Mean Squared Error (Training data): {best_mse_train}")
print(f"Mean Squared Error (Testing data): {best_mse_test}")


Best Alpha: 0.01
Mean Squared Error (Training data): 0.06762740839251773
Mean Squared Error (Testing data): 0.06840822181278264


### RandomForestRegressor model

In [24]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Step 1: Initialize variables to track the best model and MSE
best_mse_train = float('inf')
best_mse_test = float('inf')
best_n_estimators = None

# Step 2: Loop over a range of n_estimators to find the best model
for n_estimators in [10, 50, 100, 200]:  # Example values; you can adjust as needed
    # Step 3: Fit the Random Forest model
    rf_model = RandomForestRegressor(n_estimators=n_estimators, random_state=42)
    rf_model.fit(X_processed, y)

    # Step 4: Predict on training data
    y_pred_train = rf_model.predict(X_processed)

    # Step 5: Calculate MSE for training data
    mse_train = mean_squared_error(y, y_pred_train)

    # Step 6: Predict on testing data
    y_pred_test = rf_model.predict(X_test_processed)

    # Step 7: Calculate MSE for testing data
    mse_test = mean_squared_error(y_test, y_pred_test)

    # Step 8: Update the best model based on training MSE
    if mse_train < best_mse_train:
        best_mse_train = mse_train
        best_mse_test = mse_test
        best_n_estimators = n_estimators

# Step 9: Output the results
print(f"Best n_estimators: {best_n_estimators}")
print(f"Mean Squared Error (Training data): {best_mse_train}")
print(f"Mean Squared Error (Testing data): {best_mse_test}")

# Step 10: Variable importance
importances = rf_model.feature_importances_
feature_importance = dict(zip(range(X_processed.shape[1]), importances))

# Step 11: Sort and print variable importance
sorted_importance = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)
print("Feature Importance:")
for idx, importance in sorted_importance:
    print(f"Feature {idx}: {importance}")

Best n_estimators: 200
Mean Squared Error (Training data): 0.0027307095850866037
Mean Squared Error (Testing data): 0.8867148883690429
Feature Importance:
Feature 18: 0.6527160290160995
Feature 1: 0.056764685429599664
Feature 15: 0.034878899883534105
Feature 2: 0.03145367255804604
Feature 20: 0.02339870446604674
Feature 13: 0.01743509628477238
Feature 14: 0.01687899156026668
Feature 16: 0.015411688817096864
Feature 24: 0.013076656468997782
Feature 4: 0.013073624723878659
Feature 0: 0.012545539545744576
Feature 11: 0.012392659928489006
Feature 10: 0.011002763986640332
Feature 3: 0.01077408300715798
Feature 25: 0.010752755930587691
Feature 12: 0.009510815491472617
Feature 8: 0.007783158467691144
Feature 7: 0.006823371951104031
Feature 17: 0.0056193064151922895
Feature 6: 0.00433931536205768
Feature 23: 0.0042196530770236395
Feature 9: 0.0029066027099103795
Feature 29: 0.0027047268479008517
Feature 5: 0.0025919522236137926
Feature 30: 0.0015796761156941076
Feature 28: 0.001215353263386489

### MLPRegressor model

In [26]:
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, accuracy_score

# Step 2: Define the Neural Network model
nn_model = MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu', solver='adam', max_iter=200)

# Step 3: Train the model
nn_model.fit(X_processed, y)

# Step 4: Predict on training data
y_pred_train_nn = nn_model.predict(X_processed)

# Step 5: Calculate MSE for training data
mse_train_nn = mean_squared_error(y, y_pred_train_nn)
print(f"Mean Squared Error (Training data) for NN: {mse_train_nn}")

# Step 6: Predict on testing data
y_pred_test_nn = nn_model.predict(X_test_processed)

# Step 7: Calculate MSE for testing data
mse_test_nn = mean_squared_error(y_test, y_pred_test_nn)
print(f"Mean Squared Error (Testing data) for NN: {mse_test_nn}")
# Convert predictions to binary (e.g., using a threshold of 0.5)
threshold = 0.5  # Example threshold
y_pred_train_nn_binary = (y_pred_train_nn > threshold).astype(int)
y_pred_test_nn_binary = (y_pred_test_nn > threshold).astype(int)

# Assuming y11 and y22 are binary labels (0 or 1)
accuracy_train_nn = accuracy_score(y, y_pred_train_nn_binary)
accuracy_test_nn = accuracy_score(y_test, y_pred_test_nn_binary)

print(f"Accuracy (Training data) for NN: {accuracy_train_nn}")
print(f"Accuracy (Testing data) for NN: {accuracy_test_nn}")


Mean Squared Error (Training data) for NN: 0.025290086344849028
Mean Squared Error (Testing data) for NN: 0.03250845678288073
Accuracy (Training data) for NN: 0.9707016342107838
Accuracy (Testing data) for NN: 0.9627227969892431


###  variables correlation with the “loan status”. the 10most correlated and the 10 least correlated variables

In [27]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
# Ensure X is a DataFrame containing all predictor variables
train_data = pd.read_csv('trainData.csv')

# Specify columns to fill with zero
columns_to_fill_with_zero = ['mths_since_last_delinq', 'id', 'member_id']  # Adjust as per your actual column name
    
# Fill the specified columns with zero
train_data[columns_to_fill_with_zero] = train_data[columns_to_fill_with_zero].fillna(0)
    
    
# Remove rows with NaN values
train_data = train_data.dropna()
    
# Print the cleaned DataFrame shape
print("DataFrame after removing rows with NaN values:")
print(train_data.shape)
    
# Create target variable
train_data['y'] = np.where(train_data['loan_status'] == 'Charged Off', 1, 0)
    
# Separate features and target, and drop 'loan_status' column
y = train_data['y']
X = train_data.drop(['loan_status', 'y'], axis=1)

# Columns to be one-hot encoded
columns_to_encode = ['grade', 'emp_length', 'home_ownership', 'application_type']

# One-hot encode the specified columns
encoder = OneHotEncoder(drop='first', sparse_output=False)
encoded_cols = encoder.fit_transform(X[columns_to_encode])

# Create a DataFrame with the encoded columns
encoded_cols_df = pd.DataFrame(encoded_cols, columns=encoder.get_feature_names_out(columns_to_encode))

# Concatenate the original DataFrame with the encoded columns
X_encoded = pd.concat([X, encoded_cols_df], axis=1)

# Display the new DataFrame with one-hot encoded columns
print("Encoded DataFrame:")
#print(X11_encoded.head())

# Combine the predictors and the target variable into one DataFrame for correlation calculation
data_combined = pd.concat([X_encoded, y.reset_index(drop=True)], axis=1)

# Select only numeric columns for correlation calculation
numeric_cols = data_combined.select_dtypes(include=[np.number])

# Calculate the correlation matrix
correlation_matrix = numeric_cols.corr()

# Extract the correlation coefficients with the target variable
correlation_with_target = correlation_matrix[y.name].drop(labels=[y.name])

# Sort the correlation coefficients
sorted_correlations = correlation_with_target.abs().sort_values(ascending=False)

# Identify the 10 most and 10 least correlated variables
most_correlated = sorted_correlations.head(10)
least_correlated = sorted_correlations.tail(10)

# Display the results with column names
print("10 Most Correlated Variables:")
print(most_correlated)

print("\n10 Least Correlated Variables:")
print(least_correlated)

# Display the head of the DataFrame
X_encoded.head()

DataFrame after removing rows with NaN values:
(204380, 33)
Encoded DataFrame:
10 Most Correlated Variables:
grade_E                       0.118084
grade_F                       0.103126
grade_D                       0.091894
grade_B                       0.080738
grade_G                       0.058196
application_type_Joint App    0.051189
home_ownership_RENT           0.047864
home_ownership_MORTGAGE       0.043998
grade_C                       0.020276
emp_length_10+ years          0.012911
Name: y, dtype: float64

10 Least Correlated Variables:
tot_coll_amt           0.000606
loan_amnt              0.000583
annual_inc             0.000549
total_rev_hi_lim       0.000547
emp_length_5 years     0.000516
installment            0.000458
emp_length_< 1 year    0.000275
total_rec_prncp        0.000273
id                          NaN
member_id                   NaN
Name: y, dtype: float64


,id,member_id,loan_amnt,int_rate,installment,grade,emp_length,home_ownership,annual_inc,dti,...,emp_length_7 years,emp_length_8 years,emp_length_9 years,emp_length_< 1 year,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,application_type_Joint App
0,0.0,0.0,18600.0,10.99,608.86,B,6 years,RENT,80000.0,12.92,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,2000.0,17.97,72.28,D,4 years,MORTGAGE,55400.0,10.62,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,0.0,0.0,12000.0,12.29,400.24,C,10+ years,OWN,60000.0,17.92,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,16000.0,19.42,589.90,D,7 years,RENT,64000.0,3.90,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,0.0,22525.0,16.02,548.01,C,10+ years,MORTGAGE,94080.0,19.08,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
